# TGT

# TGT Backend Architecture – Developer Guide

This document explains how TGT works and how you can use and extend it from a developer perspective.

It focuses on adapting the **backend logic and linguistic processing pipeline**, not the frontend.

The frontend was almost entirely built using **Builder.io**  
https://www.builder.io/

You can adapt the frontend using Builder, but **any frontend change must respect the backend API contracts**.

In particular, you must consider:

- The API calls in `frontend/src/hooks`
- The pages using those hooks in `frontend/src/pages`
- The backend responses defined in:
  - `backend/routers`
  - `backend/app`

> All API logic, including online workers, lives in `backend/routers`.

---

## Backend Architecture

The backend is structured hierarchically using **inheritance** and **factories**.

```
backend/
├── inference/
│   ├── abstract_worker.py       ← AbstractInferenceWorker
│   ├── local_worker.py          ← LocalInferenceWorker
│   ├── processors/
│   │   ├── abstract_processor.py       ← AbstractProcessor
│   │   ├── processor_factory.py        ← ProcessorFactory
│   │   ├── labvanced/
│   │   │   ├── labvanced_base.py       ← LabvancedBaseProcessor
│   │   │   ├── transcription.py
│   │   │   ├── translation.py
│   │   │   ├── transliteration.py
│   │   │   ├── glossing.py
│   │   │   └── ColumnCreation.py
│   │   └── plain/
│   │       ├── plain_base.py           ← BasePlainProcessor
│   │       ├── transcription.py
│   │       ├── translation.py
│   │       ├── transliteration.py
│   │       └── glossing.py
│   └── strategies/
│       ├── strategy_factory.py         ← StrategyFactory
│       ├── llm_base.py                 ← LLMStrategy (Gemini/Ollama mixin)
│       ├── transcription/
│       │   ├── abstract.py             ← TranscriptionStrategy
│       │   ├── transcription_factory.py
│       │   ├── whisperx.py
│       │   ├── whisper.py
│       │   ├── bengali.py
│       │   ├── vietnamese.py
│       │   └── thai.py
│       ├── translation/
│       │   ├── abstract.py             ← TranslationStrategy
│       │   ├── translation_factory.py
│       │   ├── deepl.py
│       │   ├── llm.py
│       │   ├── marian.py
│       │   ├── M2M100.py
│       │   └── ...
│       ├── transliteration/
│       │   ├── abstract.py             ← TransliterationStrategy
│       │   ├── transliteration_factory.py
│       │   └── ...
│       └── glossing/
│           ├── abstract.py             ← GlossingStrategy
│           ├── glossing_factory.py
│           ├── llm.py
│           ├── spacy.py
│           └── ...
├── training/
├── materials/
│   └── secrets.env
└── utils/
```

> **Important:**  
> `materials/secrets.env` must contain all access tokens  
> (Azure, DeepL, HuggingFace, Google, etc.).

All components are implemented as **classes**, except utility functions inside `utils`.

- **Use inheritance to define components**
- **Use factories to instantiate components**

---

## Call Flow

A job follows this chain:

```
Worker
  └── ProcessorFactory.get_processor(format, action, language, ...)
        └── Processor (e.g. LabvancedTranslationProcessor)
              └── StrategyFactory.get_strategy(action, language, ...)
                    └── Strategy (e.g. DeepLTranslationStrategy)
```

The **Worker** orchestrates the job: iterates folders, checks cancellation, reports progress.  
The **Processor** owns file I/O: discover files → read → transform → write.  
The **Strategy** owns the model/API call for one specific task and language.

---

## Core Components

### Worker

**Role:**  
Orchestrates the job.

- Loads secrets from `materials/secrets.env`
- Builds a processor via `ProcessorFactory`
- Iterates over session folders
- Handles cancellation and progress reporting

**Type:**  
All workers extend `AbstractInferenceWorker`.

**Subclasses must implement:**
- `_initial_message()` — job-start message
- `_after_process()` — hook called after each folder completes

---

### Processor

**Role:**  
Executes the file-level job logic for a specific **format** (labvanced or plain).

- Instantiates a strategy using `StrategyFactory`
- Implements `_find_files`, `_read_file`, `_write_file`
- Implements `_process_dataframe` (the task-specific transformation)
- Manages few-shot example accumulation for LLM strategies

**Two format families:**

| Format | Base class | File target |
|--------|-----------|-------------|
| `labvanced` | `LabvancedBaseProcessor` | `trials_and_sessions_annotated.xlsx` |
| `plain` | `BasePlainProcessor` | `transcribed.xlsx` |

Each format has concrete subclasses per task: transcription, translation, glossing, transliteration.

**Type:**  
All processors extend `AbstractProcessor`.

---

### Strategy

**Role:**  
Defines how a task is performed for a specific language and provider.

Examples:

- How do we transcribe Vietnamese? → `VietnameseStrategy` (PhoWhisper)
- How do we translate German? → `DeepLTranslationStrategy`
- How do we gloss with an LLM? → `LLMGlossingStrategy` (Gemini or Qwen via Ollama)

**Task abstracts:**

- `TranscriptionStrategy`
- `TranslationStrategy`
- `TransliterationStrategy`
- `GlossingStrategy`
- `PIIStrategy`

LLM-backed strategies (translation and glossing) inherit from both the task abstract and `LLMStrategy`, which provides the Gemini/Ollama dispatch, structured JSON validation, and ID-preserving batch calls.

---

## Factories

| Factory | Input | Output |
|---------|-------|--------|
| `ProcessorFactory` | `format`, `action`, `language` | concrete `Processor` |
| `StrategyFactory` | `action`, `language` | concrete task strategy |
| `TranscriptionStrategyFactory` | `language` | concrete `TranscriptionStrategy` |
| `TranslationStrategyFactory` | `language`, `model` | concrete `TranslationStrategy` |
| `GlossingStrategyFactory` | `language`, `model` | concrete `GlossingStrategy` |
| `TransliterationStrategyFactory` | `language` | concrete `TransliterationStrategy` |

`StrategyFactory` is the unified entry point — it delegates to the task-specific factory based on `action`.  
`AbstractProcessor.__init__` calls `StrategyFactory` internally, so processors never import individual strategy classes directly.

# How the Application Runs

The application consists of two main components:

- **Frontend**: built in **TypeScript**
- **Backend**: built in **Python (FastAPI)**

The frontend is compiled into static files and automatically served by the backend.

---

## Frontend Build

To generate the frontend distribution, run:

```bash
npm run build
```

This creates a `dist/` directory containing the compiled frontend assets.

The backend (in `app.py`) is configured to **serve these static files directly**, so no separate frontend server is required in production.

---

## Backend Execution

To run the backend, navigate to the `backend/` directory and execute:

```bash
python -m uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```

### What this command does

- `python -m uvicorn` → runs the Uvicorn ASGI server  
- `app:app` → loads the FastAPI instance named `app` from `app.py`  
- `--host 127.0.0.1` → binds to localhost  
- `--port 8000` → exposes the app on port 8000  
- `--reload` → automatically restarts the server on code changes  

Once running, the application is available at:

```
http://127.0.0.1:8000
```

---

## Alternative (FastAPI CLI)

You may also run the server using FastAPI’s built-in CLI:

```bash
pip install "fastapi[standard]"
fastapi run app
```

This uses Uvicorn internally and provides the same behavior with a simpler command.


# Let's create a strategy for transcribing Thai!!

Adding a new language requires touching **two files**:

1. **Create the strategy** — add a new file under `backend/inference/strategies/transcription/`  
   (e.g. `thai.py`). Subclass `TranscriptionStrategy` and implement `load_model` and `transcribe`.

2. **Register it in the factory** — open `backend/inference/strategies/transcription/transcription_factory.py`  
   and add a new `elif` branch that returns your new strategy for the Thai language code.

That's it — the `StrategyFactory`, `ProcessorFactory`, and `Worker` pick it up automatically from there.

> **Note:** You do **not** need to create a new processor. Processors are format-specific (labvanced / plain),  
> not language-specific. The existing processors call `StrategyFactory` internally and will use your new  
> strategy as soon as it is registered in `TranscriptionStrategyFactory`.

In [9]:
%cd backend

[Errno 2] No such file or directory: 'backend'
/Users/alejandra/Documents/GitHub/TGT/backend


In [ ]:
from inference.strategies.transcription.abstract import TranscriptionStrategy

class ThaiStrategy(TranscriptionStrategy):
    def __init__(self, language_code, device = "cpu"):
        super().__init__(language_code, device)

    def load_model(self):
       pass
        
    def transcribe(self, path_to_audio: str) -> str | None:
        pass

Now we look for an appropriate model. You can look for models in the HuggingFace hub.

Let's use this https://github.com/biodatlab/thonburian-whisper

Follow the instructions on how to use the model and adjust the calls

In [ ]:
from transformers import pipeline
from inference.strategies.transcription.abstract import TranscriptionStrategy
import torch

class ThaiStrategy(TranscriptionStrategy):

    def load_model(self):
        MODEL_NAME = "biodatlab/whisper-th-large-v3-combined"

        device = 0 if torch.cuda.is_available() else "cpu"

        self.pipe = pipeline(
            task="automatic-speech-recognition",
            model=MODEL_NAME,
            chunk_length_s=30,
            device=device,
        )
        
    def transcribe(self, path_to_audio: str) -> str | None:
        self.pipe.model.config.forced_decoder_ids = self.pipe.tokenizer.get_decoder_prompt_ids(
        language=self.language_code,
        task="transcribe"
        )
        text = self.pipe(path_to_audio)["text"]
        return text

Now we need to modify the factory to make sure this strategy is called when thai is the specified language

In [ ]:
from inference.strategies.transcription.abstract import TranscriptionStrategy
from inference.strategies.transcription.whisperx import WhisperxStrategy
from inference.strategies.transcription.whisper import WhisperStrategy
from inference.strategies.transcription.bengali import BengaliStrategy
from inference.strategies.transcription.vietnamese import VietnameseStrategy
from inference.strategies.transcription.thai import ThaiTranscriptionStrategy  # this is new


class TranscriptionStrategyFactory:
    @staticmethod
    def get_strategy(language_code: str) -> TranscriptionStrategy:
        if language_code in ['en', 'fr', 'de', 'es', 'it', 'ja', 'zh', 'nl',
                             'uk', 'pt', 'ar', 'cs', 'ru', 'pl', 'hu', 'fi',
                             'fa', 'el', 'tr', 'da', 'he', 'ko', 'ur', 
                             'te', 'hi', 'ca', 'ml', 'ro', 'ka', 'tl']:
            return WhisperxStrategy(language_code)
        elif language_code in ['vi']:
            return VietnameseStrategy(language_code)
        elif language_code in ['et']:
            return WhisperStrategy(language_code)
        elif language_code in ['bn']:
            return BengaliStrategy(language_code)
        elif language_code in ['th']:  # this is also new
            return ThaiTranscriptionStrategy(language_code)
        else:
            raise ValueError(f"No transcription strategy available for language code: {language_code}")

It is done! Now lets call the app and see how it works!

Important notes:

- If some library is not included, you need to be sure to install libraries and be careful of potential conflicts.
- Note that this is retrieving the model from huggingface and therefore you need to have an HuggingFace token under secrets


In [16]:
# Inside the Backend repository!!
!python -m uvicorn app:app --host 127.0.0.1 --port 8000 --reload

INFO:     Will watch for changes in these directories: ['/Users/alejandra/Documents/GitHub/TGT/backend']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [12340] using WatchFiles
INFO:     Started server process [12342]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     127.0.0.1:59837 - "GET /api/inference/models/translation HTTP/1.1" 200 OK
Loading token cache from /Users/alejandra/Documents/GitHub/TGT/backend/materials/token_cache.json
Account(s) in cache: camelo.cruz@leibniz-zas.de
Using cached token, no login required.
Saving token cache to /Users/alejandra/Documents/GitHub/TGT/backend/materials/token_cache.json
INFO:     127.0.0.1:59837 - "GET /api/auth/start HTTP/1.1" 302 Found
INFO:     127.0.0.1:59840 - "GET /api/inference/models/translation HTTP/1.1" 200 OK
Loading token cache from /Users/alejandra/Documents/GitHub/TGT/backend/materials/token_cache.json
Account(s) in cache: c